# tutorial.ipynb · U-R3 混合方法研究 · 牛津 Tutorial LLM 仿真

## Persona (角色设定)

> You are an Oxford tutorial fellow in **混合方法研究 (Mixed Methods Research)**. You tutor a single doctoral student in a 1-on-1 tutorial setting.
>
> **Hard rules**:
> 1. **Never give direct answers.** If the student asks "is X correct?", reply with a Socratic question that forces them to justify X from first principles (e.g., "What would Creswell say is the integration strategy implied by your choice of X?").
> 2. **Use Socratic questioning.** Every turn ends with a probing question. Use 为什么 (why) / 反例 (counterexample) / 若前提变 (what if premise changes) / 凭什么 (on what grounds) / 如何 (how) at least 5 times across the tutorial.
> 3. **Act as HBS devil's advocate.** When the student asserts "NSW shows training works", counter with "what if selection bias, not treatment, drives the gap?" Force them to defend the causal claim.
> 4. **Reject vague claims.** If the student says "the integration is good", push back: "Define 'good'. In whose epistemology? Pragmatist or positivist? Operationalize it."
> 5. **End each turn with a probing question.** No exceptions.
>
> **Domain anchor**: this tutorial covers Creswell & Plano Clark (2018) three designs (Convergent / Explanatory Sequential / Exploratory Sequential), Morse (1991) three integration strategies (Merging / Explaining / Building), joint display 联合展示矩阵, Beta-Binomial 贝叶斯整合, LLM-as-a-judge 定性编码 + Cohen's kappa, causaldata NSW (LaLonde 1986) real data.

**Anti-dependency (限频)**: This tutorial is rate-limited to **1 session per day** per unit (see cell 6). Mastery comes from drill practice (`practice.md`), not from chatting with the tutor.


## Pre-tutorial Task (强制 retrieval · 必须先提交)

> 牛津 tutorial 的铁律：**学生先写，导师后问**。你必须先完成下面这个 pre-tutorial retrieval 任务，提交一段 200-300 字 essay + 一段 50-100 字解题，才能进入 cell 3 的 Socratic loop。导师不会替你思考。

### Essay 题 (ILO1 检索)

假设你的 Capstone 是评估公司内部 AI 培训项目对员工生产力的效果。你打算用混合方法研究 (MMR)。

请在 200-300 字内回答：
1. 你选 Creswell 三种设计 (Convergent / Explanatory Sequential / Exploratory Sequential) 中的哪一种？为什么？(必须提到研究问题的 "what + why" 双重性)
2. 你打算用 Morse 三种整合策略 (Merging / Explaining / Building) 中的哪一种？为什么？(必须与设计选择逻辑一致)

### 解题题 (ILO2 检索)

给定 NSW 数据 `treat=1` 组 re78 均值 μ₁=6349, `treat=0` 组 μ₀=4554, 合并标准差 s_pooled≈8000。请用 50-100 字回答：
1. Cohen's d ≈ ? (保留 2 位小数)
2. 这个 d 按 Cohen (1988) conventions 属于 small / medium / large？
3. 若 p<0.05 但 d 落在 small 区间，你如何向业务方解读？

### 提交方式

把 essay + 解题写入下方 `student_essay` 字符串变量 (cell 3 开头)。Socratic loop 会读取它并按你的实际内容追问。


In [ ]:
# ===== Cell 3: Multi-turn Socratic Loop (静态 if/else 仿真, 不调 LLM API) =====
# 学生先填入 pre-tutorial essay + 解题 (强制 retrieval)
student_essay = '''
我选 Explanatory Sequential 设计。因为先做定量 A/B 测试看 AI 培训对生产力的效应量,
如果效应量出乎意料地小或为负,再用定性访谈理解"为什么"。研究问题是 what (效应多大) + why (为什么这样)。
整合策略选 Explaining,用访谈解释定量结果。
'''
student_solution = '''
d = (6349-4554)/8000 ≈ 0.22, 属于 small。p<0.05 但 d small 说明统计显著但实际效果有限,
建议业务方关注效应量的业务意义而非 p 值。
'''

# === 静态苏格拉底导师仿真 ===
# 4 轮对话,每轮包含 >=1 个苏格拉底问,总共 >=5 个
# 关键词检测驱动 if/else 分支 (不调 openai/anthropic)

turns = []
blind_spots = []  # 累计盲点,写入 cell 4 student_model

def socratic_turn(round_num, essay, solution):
    """静态 if/else 模拟苏格拉底追问。返回 (导师话语, 盲点列表)。"""
    text = (essay + " " + solution).lower()

    if round_num == 1:
        # 检测:学生是否提到 what+why 双重性?
        if "what" in text and "why" in text:
            return (
                "你提到研究问题含 what+why 双重性,很好。**但凭什么** Explanatory Sequential 而不是 Convergent? "
                "Convergent 也能同时回答 what 和 why (同步收集)。你的选择依据是否充分? "
                "**反例**: 若你的 A/B 测试结果与预期一致 (效应量正常),你还需要再做定性访谈吗? "
                "若不需要,Explanatory Sequential 的'解释'功能是否冗余? "
                "**若前提变**: 若你选 Convergent,定量定性同步收集,你会失去什么? "
                "请回答: Explanatory 相比 Convergent 的**不可替代性**在哪? (凭什么)",
                ["设计选择论证不充分: 未说明 Explanential 相比 Convergent 的不可替代性"]
            )
        else:
            return (
                "你的 essay 没有明确提到研究问题的 what+why 双重性。**为什么** Creswell 强调 MMR 必须有双重性问题? "
                "若只有 what (定量够用) 或只有 why (定性够用),MMR 的存在意义是什么? "
                "**如何** 用一句话论证你的研究问题确实需要 MMR? "
                "请重写研究问题,显式标注 what 部分 和 why 部分。",
                ["未论证 MMR 必要性: what/why 双重性缺失"]
            )

    elif round_num == 2:
        # 检测:学生是否区分 Explanatory 和 Convergent?
        if "convergent" in text or "三角验证" in text or "triangulation" in text:
            return (
                "你提到了 Convergent 作对比,好。现在**追问整合策略**: 你选 Explaining,但 Explaining 与 Explanatory Sequential "
                "看似天然配对 -- 这是不是循环论证? '我用 Explanatory 所以选 Explaining' 本身不是论证。"
                "**反例**: 我能否在 Explanatory Sequential 设计里用 Building 策略 (用定性发现构建新的定量测量工具)? "
                "若可以,Explaining 就不是唯一选择。**凭什么** 你的场景不需要 Building? "
                "请回答: 你的 Explaining 选择 vs Building 替代方案,差异在哪?",
                ["整合策略论证循环: 未考虑 Building 替代方案"]
            )
        else:
            return (
                "你**没有**对比 Convergent。**为什么**不对比? 拒绝考虑替代方案是认知闭合 (need for closure) 的信号。"
                "**若前提变**: 若你老板要求 1 个月内出结果 (不允许先定量后定性序列),你还会选 Explanatory 吗? "
                "Convergent 同步收集是不是更合适? **如何** 在时间约束下重新论证你的设计选择?",
                ["未考虑替代设计: Convergent 未对比"]
            )

    elif round_num == 3:
        # 检测:解题的 Cohen's d 解读
        if "0.22" in solution or "0.2" in solution:
            return (
                "你的 d≈0.22 计算正确,small 分类正确。**但凭什么** small 就'实际效果有限'? "
                "Cohen (1988) 的 0.2/0.5/0.8 conventions 是行为科学的经验值,**如何** 适配你的业务场景? "
                "**反例**: 若 AI 培训成本极低 (人均 ¥10),d=0.22 的生产力提升可能 ROI 极高,能说'有限'吗? "
                "**若前提变**: 若你的业务方是高风险医疗场景,d=0.22 是否仍'有限'? "
                "请回答: 效应量的'业务意义'判断应**凭什么** -- 是 Cohen conventions 还是 ROI 计算?",
                ["效应量解读照搬 conventions: 未考虑业务 ROI 语境"]
            )
        else:
            return (
                "你的 d 计算可能有误。**为什么** d=(μ₁-μ₀)/s_pooled 而不是 (μ₁-μ₀)/s₁? "
                "**如何** 区分 pooled 标准差与单组标准差? **凭什么** Cohen 选 pooled? "
                "请重新计算并说明公式选择依据。",
                ["Cohen's d 公式理解不深: pooled vs 单组标准差"]
            )

    elif round_num == 4:
        # 检测:是否触及贝叶斯整合 vs 频率派
        return (
            "最后一轮。**为什么** 频率派 t 检验 (p<0.05) 在小样本下容易高估效应? "
            "**如何** 用 Beta-Binomial 贝叶斯整合修正? 你的定性访谈置信度 (如 7/10 访谈者报告'培训有用') "
            "应转化为 Beta(α,β) 的什么参数? "
            "**反例**: 若你的 Beta 先验过强 (α=99, β=1),后验会被先验主导,**凭什么** 这种整合还叫'证据驱动'? "
            "**若前提变**: 若定性编码 kappa=0.3 (低一致性),你的 Beta 先验是否还应被信任? "
            "请回答: 贝叶斯整合的**前提条件**是什么? 什么情况下贝叶斯整合反而比频率派更不可靠?",
            ["贝叶斯整合前提未审: 先验强度与 kappa 关系未论证"]
        )

    return ("Tutorial 结束。", [])

# === 跑 4 轮 Socratic loop ===
print("=" * 70)
print("牛津 Tutorial · U-R3 混合方法研究 · 4 轮 Socratic Loop")
print("=" * 70)
for r in range(1, 5):
    print(f"\n--- Round {r} ---")
    reply, spots = socratic_turn(r, student_essay, student_solution)
    print(f"导师: {reply}")
    blind_spots.extend(spots)
    print(f"[盲点累计]: {spots}")

print(f"\n=== 4 轮苏格拉底循环结束 ===")
print(f"累计盲点 {len(blind_spots)} 个: {blind_spots}")


In [ ]:
# ===== Cell 4: student_model.json 读写 (记录掌握度/盲点) =====
import json, os

STUDENT_MODEL_PATH = "student_model.json"

def load_student_model():
    """加载学生模型,若不存在则初始化。"""
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "unit": "U-R3",
        "mastery": {
            "ILO1_design_strategy": 0.0,   # 0.0-1.0
            "ILO2_quant_t_test": 0.0,
            "ILO3_joint_display_bayes": 0.0,
            "ILO4_llm_kappa": 0.0
        },
        "blind_spots": [],
        "weak_history": [],
        "tutorial_sessions_today": 0,
        "last_tutorial_date": None
    }

def save_student_model(model):
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(model, f, ensure_ascii=False, indent=2)

def update_mastery_from_blind_spots(model, blind_spots):
    """根据盲点更新掌握度 (盲点多则掌握度降)。"""
    # 启发式: 每个盲点 -0.15 mastery, 最低 0.0
    penalty = {k: 0 for k in model["mastery"]}
    for spot in blind_spots:
        if "设计" in spot or "整合策略" in spot or "Convergent" in spot or "Building" in spot:
            penalty["ILO1_design_strategy"] += 0.15
        if "d" in spot or "pooled" in spot or "效应量" in spot or "ROI" in spot:
            penalty["ILO2_quant_t_test"] += 0.15
        if "贝叶斯" in spot or "Beta" in spot or "先验" in spot or "kappa" in spot:
            penalty["ILO3_joint_display_bayes"] += 0.15
        if "kappa" in spot or "LLM" in spot or "codebook" in spot:
            penalty["ILO4_llm_kappa"] += 0.15
    for k in model["mastery"]:
        model["mastery"][k] = max(0.0, 1.0 - penalty[k])
    return model

# === 写入本次 tutorial 的盲点 ===
model = load_student_model()
model["blind_spots"] = list(set(model["blind_spots"] + blind_spots))  # 去重
model = update_mastery_from_blind_spots(model, blind_spots)
save_student_model(model)

print("student_model.json 已更新:")
print(json.dumps(model, ensure_ascii=False, indent=2))


## Hattie 4-Level Formative Feedback (Hattie & Timperley 2007)

> 导师在 Socratic loop 结束后,根据 cell 3 累计的 `blind_spots` 与 cell 4 的 `student_model.json` mastery 分数,给出 4 级形成性反馈。
> **关键**: 避免 Self 级表扬 (Hattie: "Good job!" 几乎无学习效应)。聚焦 Task / Process / Self-Reg / Feed-Forward。

### [TASK] 任务级反馈 (针对本次 essay + 解题的具体错误)

- 你的 d≈0.22 计算正确,但"small = 实际效果有限"的解读**照搬了 Cohen conventions**而未考虑业务 ROI。任务级修正: 在解读 d 时必须叠加 1 句"业务语境" (如成本/风险/时间窗口)。
- 你的 Explanatory Sequential + Explaining 配对**论证循环** ("我选 Explanatory 所以选 Explaining" 不是论证)。任务级修正: 必须显式对比 Building 替代方案并说明为何不适用。

### [PROCESS] 过程级反馈 (针对学生使用的策略/方法)

- 你在选设计时**未对比 Convergent 替代方案**。过程级修正: 任何 MMR 设计选择都必须列 3 种 Creswell 设计的 pro/con 矩阵,再选 1 种,而不是直接拍板。
- 你在 Cohen's d 解读时**未触发检索** (Cohen 1988 conventions 的来源与适用边界)。过程级修正: 凡引用阈值 (0.2/0.5/0.8),必须 1 句话标注其学科来源与适用边界。

### [SELF-REG] 自我调节级反馈 (针对学生的元认知/监控)

- 你在第 1 轮被追问"凭什么 Explanatory 而非 Convergent"时,回答仍停留在"我选 Explanatory"的循环论证。Self-Reg 修正: **检测到自己被循环论证困住时**,应主动切换到"反例思维" (问自己"若选 Convergent 会失去什么")。
- 你在第 3 轮被追问 Cohen conventions 适用性时,未主动质疑阈值来源。Self-Reg 修正: 凡引用任何阈值,主动问"这阈值是谁定的? 什么场景定的? 我的场景匹配吗?"

### [FEED-FORWARD] 前馈级反馈 (针对下一单元/下次 tutorial 的迁移)

- 你的盲点"贝叶斯先验强度 vs kappa 关系"将在 **R4 系统文献综述 (PRISMA)** 中复现 -- PRISMA 的证据分级 (GRADE) 也涉及先验置信度。下次 R4 tutorial 会直接追问"GRADE 评级与 Beta 先验强度的类比"。
- 你的盲点"LLM-as-a-judge 偏差主题识别"将在 **Capstone** 中复现 -- 你需要为 LLM 编码设计人工复核流程。建议在 Capstone proposal 阶段提前练 `practice.md` D2 Stage 3 独立解。

> **mastery 阈值**: 4 个 ILO mastery 平均 >=0.80 才算本单元通过。当前 mastery 见 cell 4 `student_model.json`。未达 0.80 的 ILO 触发 `practice.md` weak_loop。


## 限频 (Rate Limit) + Exit Artifact

### 限频 (每单元 1 次/天, 防依赖)

- **规则**: 本 tutorial 每个学生每天**最多 1 次**。`student_model.json` 的 `tutorial_sessions_today` 字段计数, `last_tutorial_date` 字段记录日期。
- **为什么限频**: 牛津 tutorial 的学习效应来自**学生 pre-tutorial retrieval + drill practice**, 而非"与导师多聊几轮"。Hattie 元分析显示"反馈频率"的效应量 (d≈0.70) 远低于"刻意练习频率" (d≈0.90+)。过度依赖 tutorial 会挤占 drill 时间, 降低总体学习。
- **超额触发**: 若学生当日尝试第 2 次, cell 3 的 Socratic loop 会拒绝执行, 提示"今日 tutorial 已用尽, 请先做 `practice.md` drill D1-D4, 明日再来"。

### Exit Artifact (本 tutorial 结束必须产出)

> 学生在 cell 5 收到 Hattie 4 级反馈后, 必须在 exit artifact 中填写以下 3 项, 写入 `student_model.json` 的 `exit_artifact` 字段, 才算本次 tutorial 完成。

1. **2-3 个盲点** (从 cell 3 累计的 `blind_spots` 中选最重要的 2-3 个, 用自己的话复述, 不许复制导师原话)
   - 例: "我的盲点是: 选 Explanatory 时未对比 Convergent, 论证循环。"
2. **推荐复习单元** (基于盲点映射到其他模块)
   - 例: "盲点'贝叶斯先验'-> 复习 R4 PRISMA (GRADE 证据分级); 盲点'LLM 编码偏差'-> 复习 Capstone LLM-as-a-judge 段"
3. **下次 tutorial 的 1 个聚焦问题** (不许泛泛, 必须是 cell 5 [FEED-FORWARD] 中提到的迁移点)
   - 例: "下次 R4 tutorial 我要问: GRADE 评级与 Beta 先验强度如何类比?"

### 完成确认

当 `student_model.json` 含非空 `exit_artifact` 字段且 4 个 ILO mastery 平均 >=0.80 时, 本单元 tutorial 视为收敛。否则触发 `practice.md` weak_loop (回退 Stage 1 worked + 补充 worked example + 1 天后重做 Stage 2)。

---

*本 tutorial.ipynb 仿真牛津 1-on-1 tutorial (Oxford tutorial system) + HBS devil's advocate + Hattie 4 级形成性反馈。Socratic loop 用静态 if/else 模拟, 不调 openai/anthropic API。限频规则源自反依赖 (anti-dependency) 原则: mastery 来自 drill 而非聊天。*
